# Fase 1 - Analisis Exploratorio de Datos (EDA)

**Proyecto:** Priorizacion de brigadas de salud en comunidades rurales de Colombia.

**Objetivo:** cargar el dataset simulado, describirlo, visualizar sus variables y detectar problemas de calidad (nulos, outliers, desbalance de clases, rangos) para preparar la fase de modelado.

**Fuente:** `output/dataset_completo.csv` (37 columnas). La version lista para modelado `dataset_ml.csv` se menciona al final.

## 0. Configuracion

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

print('Librerias cargadas correctamente')

## 1. Carga y descripcion del dataset

En Google Colab, ejecuta esta celda: si el archivo no esta en el directorio de trabajo, te pedira subir `dataset_completo.csv`.

In [ ]:
RUTA = 'dataset_completo.csv'

if not os.path.exists(RUTA):
    try:
        from google.colab import files
        subidos = files.upload()
        RUTA = next(iter(subidos))
    except Exception as exc:
        raise SystemExit('No se encontro el archivo. Sube dataset_completo.csv') from exc

df = pd.read_csv(RUTA, encoding='utf-8-sig')

print('Archivo:', RUTA)
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])

In [ ]:
print('Tipos de datos por columna:')
print(df.dtypes.to_string())

print()
print('Primeras 5 filas:')
df.head()

### 1.1 Diccionario de columnas

| Grupo | Columnas |
|---|---|
| Identificacion | `comunidad_id` |
| Geografia | `departamento`, `municipio`, `zona_geografica`, `altitud_msnm` |
| Comunidad | `tipo_comunidad` |
| Demografia | `poblacion_total`, `menores_5`, `adultos_mayores`, `gestantes`, `personas_discapacidad` |
| Acceso / logistica | `distancia_km`, `tiempo_acceso_min`, `acceso_vial`, `transporte_disponible` |
| Servicios | `agua_potable`, `alcantarillado`, `puesto_salud_cercano` |
| Salud | `cobertura_salud_pct`, `cobertura_vacunacion_pct`, `casos_prioritarios_30d`, `enfermedades_cronicas`, `alerta_epidemiologica`, `ultima_brigada_dias`, `brigadas_ultimos_12m`, `demanda_insatisfecha_pct` |
| Ambiente / conectividad | `conectividad`, `temporada`, `riesgo_inundacion`, `riesgo_deslizamiento` |
| Indices (intermedios) | `indice_necesidad`, `indice_acceso`, `indice_vulnerabilidad`, `indice_brecha` |
| Objetivos | `puntaje_prioridad` (regresion), `prioridad` (clasificacion), `split` |

In [ ]:
numericas = df.select_dtypes(include=[np.number])
print('Estadisticas descriptivas de variables numericas:')
numericas.describe().T.round(2)

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns
for col in categoricas:
    print('===', col, '(', df[col].nunique(), 'categorias )===')
    print(df[col].value_counts().to_string())
    print()

## 2. Estadisticas descriptivas y visualizaciones

In [ ]:
cols_num = df.select_dtypes(include=[np.number]).columns.tolist()

df[cols_num].hist(bins=30, figsize=(16, 14), edgecolor='black')
plt.suptitle('Distribucion de variables numericas', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
indices = ['indice_necesidad', 'indice_acceso', 'indice_vulnerabilidad',
           'indice_brecha', 'puntaje_prioridad']

fig, axes = plt.subplots(1, len(indices), figsize=(18, 5))
for ax, col in zip(axes, indices):
    sns.boxplot(y=df[col], ax=ax, color='#4C72B0')
    ax.set_title(col, fontsize=10)
plt.suptitle('Boxplots de indices y puntaje de prioridad', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
orden = ['BAJA', 'MEDIA', 'ALTA', 'CRÍTICA']

sns.countplot(data=df, x='prioridad', order=orden, hue='prioridad', legend=False, palette='viridis')
plt.title('Distribucion de la clase prioridad')
plt.xlabel('Prioridad')
plt.ylabel('Numero de comunidades')
plt.show()

In [ ]:
cats = ['zona_geografica', 'tipo_comunidad', 'acceso_vial', 'conectividad', 'temporada']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), cats):
    sns.countplot(data=df, y=col, ax=ax, hue=col, legend=False,
                  order=df[col].value_counts().index, palette='mako')
    ax.set_title(col)
    ax.set_xlabel('')
for ax in axes.ravel()[len(cats):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
corr = df[cols_num].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Matriz de correlacion (variables numericas)')
plt.show()

In [ ]:
muestra = df.sample(1500, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col in zip(axes, indices[:4]):
    sns.scatterplot(data=muestra, x=col, y='puntaje_prioridad', ax=ax, alpha=0.3)
    ax.set_title(col + ' vs puntaje')
plt.tight_layout()
plt.show()

In [ ]:
umbrales = {'BAJA': 42, 'MEDIA': 54, 'ALTA': 66}

sns.histplot(df['puntaje_prioridad'], bins=40, kde=True, color='#4C72B0')
for valor in umbrales.values():
    plt.axvline(valor, color='red', linestyle='--', alpha=0.7)
plt.title('Distribucion de puntaje_prioridad con umbrales (42 / 54 / 66)')
plt.xlabel('puntaje_prioridad')
plt.show()

## 3. Calidad de datos

In [ ]:
nulos = df.isna().sum()
print('Valores nulos por columna:')
if nulos.any():
    print(nulos[nulos > 0].to_string())
else:
    print('  (ninguno)')

print()
print('Filas duplicadas:', df.duplicated().sum())
print('IDs duplicados:', df['comunidad_id'].duplicated().sum())

In [ ]:
resumen = []
for col in cols_num:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < li) | (df[col] > ls)).sum())
    resumen.append((col, n_out, round(n_out / len(df) * 100, 2), round(li, 2), round(ls, 2)))

outliers = pd.DataFrame(resumen, columns=['columna', 'n_outliers', 'pct', 'lim_inf', 'lim_sup'])
print('Outliers por el metodo IQR (1.5):')
outliers.sort_values('n_outliers', ascending=False).reset_index(drop=True)

In [ ]:
conteo = df['prioridad'].value_counts()
porcentaje = (conteo / len(df) * 100).round(2)
tabla = pd.DataFrame({'conteo': conteo, 'porcentaje': porcentaje})
print('Balance de clases de prioridad:')
print(tabla.to_string())
print()
print('Ratio clase mayor/menor:', round(conteo.max() / conteo.min(), 2))

In [ ]:
binarias = ['agua_potable', 'alcantarillado', 'puesto_salud_cercano', 'alerta_epidemiologica']
for col in binarias:
    print(col, '->', sorted(df[col].unique()))

print()
print('Rangos observados:')
print('cobertura_salud_pct     :', df['cobertura_salud_pct'].min(), '-', df['cobertura_salud_pct'].max())
print('cobertura_vacunacion_pct:', df['cobertura_vacunacion_pct'].min(), '-', df['cobertura_vacunacion_pct'].max())
print('riesgo_inundacion       :', df['riesgo_inundacion'].min(), '-', df['riesgo_inundacion'].max())
print('puntaje_prioridad       :', df['puntaje_prioridad'].min(), '-', df['puntaje_prioridad'].max())

In [ ]:
print('Registros por split:')
print(df['split'].value_counts().to_string())

## 4. Conclusiones

- El dataset no presenta valores nulos ni filas duplicadas.
- Las variables numericas tienen rangos coherentes con su definicion.
- La clase `prioridad` es ordinal y moderadamente desbalanceada; conviene reportar metricas macro y tenerlo en cuenta al modelar.
- La version para modelado es `dataset_ml.csv`: 29 features + `puntaje_prioridad` (regresion) + `prioridad` + `split`. Los indices `indice_*` y `comunidad_id` se excluyen para evitar fuga de informacion.
- Siguiente fase: entrenar la red feedforward (29 entradas -> `puntaje_prioridad`) y derivar la clase con los umbrales 42 / 54 / 66.